In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o "/content/drive/MyDrive/waste_project_2/garbage.zip" -d /content/data

import torch, random, numpy as np, torch.nn as nn
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

root = Path("/content/data")
classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(classes)}
paths, labels = [], []
for c in classes:
    for p in (root/c).glob("*"):
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            paths.append(p); labels.append(class_to_idx[c])

X_train, X_tmp, y_train, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(classes, len(X_train), len(X_val), len(X_test))

IMG = 128
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
test_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(),
    transforms.Normalize(mean, std)])

class WasteDS(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]

train_dl = DataLoader(WasteDS(X_train, y_train, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(WasteDS(X_val, y_val, test_tf), batch_size=32, num_workers=2)
test_dl = DataLoader(WasteDS(X_test, y_test, test_tf), batch_size=32, num_workers=2)

def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1),
        nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))

class WasteCNN(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.4), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

def run_epoch(dl, train):
    model.train(train)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                optimizer.step(); scheduler.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

counts = np.bincount(y_train, minlength=len(classes))
weights = torch.tensor(counts.sum() / (len(classes) * counts), dtype=torch.float).to(device)


In [ ]:
EPOCHS = 40
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=len(train_dl))

def run_epoch(dl, train):
    model.train(train)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

SAVE = "/content/drive/MyDrive/waste_project_2/best_cnn.pt"
best_acc, patience, wait = 0, 8, 0
history = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}

for ep in range(EPOCHS):
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    for k, v in zip(history, [tl, ta, vl, va]):
        history[k].append(v)
    print(f"Epoch {ep+1}: train acc {ta:.3f} | val acc {va:.3f} | val loss {vl:.3f}")
    if va > best_acc:
        best_acc, wait = va, 0
        torch.save(model.state_dict(), SAVE)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break
print("Best val acc:", best_acc)

In [ ]:
EPOCHS = 80
counts = np.bincount(y_train, minlength=len(classes))
weights = torch.tensor(counts.sum() / (len(classes) * counts), dtype=torch.float).to(device)

model = WasteCNN(len(classes)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=len(train_dl))

SAVE = "/content/drive/MyDrive/waste_project_2/best_cnn_v2.pt"
best_acc, patience, wait = 0, 15, 0

for ep in range(EPOCHS):
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    print(f"Epoch {ep+1}: train acc {ta:.3f} | val acc {va:.3f} | val loss {vl:.3f}")
    if va > best_acc:
        best_acc, wait = va, 0
        torch.save(model.state_dict(), SAVE)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break
print("Best val acc:", best_acc)

In [ ]:
# check: tera scratch CNN dobara load karke test accuracy
model = WasteCNN(len(classes)).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/waste_project_2/best_cnn_v2.pt"))
model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        trues += y.tolist()
print("CNN v2 TEST ACCURACY:", round(accuracy_score(trues, preds) * 100, 2), "%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Load the saved best model (v2) and switch to evaluation mode
model = WasteCNN(len(classes)).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/waste_project_2/best_cnn_v2.pt"))
model.eval()  # turns off dropout, so predictions are stable

# Predict on the test set (images the model never saw during training)
preds, trues = [], []
with torch.no_grad():  # no gradients needed, saves memory and time
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()  # class with highest score = prediction
        trues += y.tolist()                                    # the real labels

# Precision, recall and F1 for each class
print(classification_report(trues, preds, target_names=classes))

# Confusion matrix: rows = actual class, columns = predicted class
cm = confusion_matrix(trues, preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - WasteCNN v2")

# Save as an image for the GitHub README
plt.savefig("/content/drive/MyDrive/waste_project_2/confusion_matrix_v2.png", dpi=150, bbox_inches="tight")
plt.show()